In [ ]:
# read pickle file
import pandas as pd

df8918 = pd.read_pickle('******.pkl')
df8918


In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918_exacerbated_0_1 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918_exacerbated_0_1

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_exacerbated_0_1.copy()

# features, emotion_labels, speaker_ids defined
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)

In [ ]:
# Step 1: Divide the dataset into a test set (20%), and use the remaining 80% for training and validation, dividing it according to the distribution of y.
X_temp, X_test, y_temp, y_emo_test, sid_temp, y_spk_test = train_test_split(
    features, emotion_labels, speaker_labels, test_size=0.2, stratify=emotion_labels, random_state=42
)   
# Step 2: Divide the remaining 80% into a validation set (25% of 80% = 20%) and a training set (75% of 80% = 60%), again dividing them according to the distribution of y.
X_train, X_val, y_emo_train, y_emo_val, y_spk_train, y_spk_val = train_test_split(
    X_temp, y_temp, sid_temp, test_size=0.2, stratify=y_temp, random_state=42
)   

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, features, emotion_labels, speaker_labels):
        self.features = torch.tensor(features)
        self.emotion_labels = torch.tensor(emotion_labels)
        self.speaker_labels = torch.tensor(speaker_labels)
    def __len__(self): return len(self.emotion_labels)
    def __getitem__(self, idx): return self.features[idx], self.emotion_labels[idx], self.speaker_labels[idx]

train_loader = DataLoader(MultiTaskDataset(X_train, y_emo_train, y_spk_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, y_emo_val, y_spk_val), batch_size=32, shuffle=False)
test_loader = DataLoader(MultiTaskDataset(X_test, y_emo_test, y_spk_test), batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# GRL（Use only in adversarial models）
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_param=1.0):
        super().__init__()
        self.lambda_param = lambda_param
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_param)

In [ ]:
def compute_j_ratio(embeddings, labels):
    classes = np.unique(labels)
    if len(classes) < 2: return 0.0
    D = embeddings.shape[1]
    overall_mean = np.mean(embeddings, axis=0)
    S_B, S_W = np.zeros((D, D)), np.zeros((D, D))
    for c in classes:
        X_c = embeddings[labels == c]
        mean_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]
        diff = (mean_c - overall_mean).reshape(-1, 1)
        S_B += n_c * diff @ diff.T
        cov = np.cov(X_c, rowvar=False)
        S_W += (n_c - 1) * cov
    return np.trace(S_B) / (np.trace(S_W) + 1e-8)

def extract_embeddings(model, loader, device):
    model.eval()
    embs, spks = [], []
    with torch.no_grad():
        for feats, _, spk in loader:
            feats = feats.to(device)
            emb = model(feats)
            embs.append(emb.cpu().numpy())
            spks.append(spk.numpy())
    return np.concatenate(embs, 0), np.concatenate(spks, 0)

In [ ]:
# ========= Model definition (shared) =========
class LocallyEnhancedFFN(nn.Module):
   def __init__(self, input_dim):
       super().__init__()
       self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=1)
       self.depthwise = nn.Conv1d(64, 64, kernel_size=1, groups=64)
       self.conv2 = nn.Conv1d(64, input_dim, kernel_size=1)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       residual = x
       x = x.permute(0, 2, 1)
       x = self.conv1(x)
       x = self.depthwise(x)
       x = F.gelu(x)
       x = self.dropout(x)
       x = self.conv2(x)
       x = F.gelu(x)
       x = x.permute(0, 2, 1)
       return x + residual

class TransformerEncoderBlock(nn.Module):
   def __init__(self, embed_dim, num_heads):
       super().__init__()
       self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.5)
       self.norm1 = nn.LayerNorm(embed_dim)
       self.norm2 = nn.LayerNorm(embed_dim)
       self.ffn = LocallyEnhancedFFN(embed_dim)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       residual = x
       x_attn = x.permute(1, 0, 2)
       attn_output, _ = self.self_attn(x_attn, x_attn, x_attn)
       attn_output = attn_output.permute(1, 0, 2)
       attn_output = self.dropout(attn_output)
       x = self.norm1(residual + attn_output)
       ffn_output = self.ffn(x)
       return self.norm2(x + ffn_output)

class UpstreamModel(nn.Module):
   def __init__(self, input_dim=54, hidden_dim=128, embed_dim=64):
       super().__init__()
       self.reshape = lambda x: x.unsqueeze(1)
       self.encoder1 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
       self.encoder2 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
       self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim//2, bidirectional=True, batch_first=True)
       self.fc = nn.Linear(hidden_dim, embed_dim)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       x = self.reshape(x)
       x = self.encoder1(x)
       x = self.encoder2(x)
       x, _ = self.lstm(x)
       x = x.squeeze(1)
       x = self.fc(x)
       x = self.dropout(x)
       return x

class EmotionClassifier(nn.Module):
   def __init__(self, embed_dim=64):
       super().__init__()
       self.classifier = nn.Sequential(
           nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(64,2)
       )
   def forward(self, x):
       return self.classifier(x)

class SpeakerClassifier(nn.Module):
   def __init__(self, embed_dim, num_speakers):
       super().__init__()
       self.grl = GradientReversalLayer()  # use GRL
       self.classifier = nn.Sequential(
           nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(64, num_speakers)
       )
   def forward(self, x):
       x = self.grl(x)  # Gradient Reversal
       return self.classifier(x)


def evaluate_emotion(upstream, cls, loader):
   upstream.eval(); cls.eval()
   probs, labels = [], []
   with torch.no_grad():
       for feats, emo, _ in loader:
           feats = feats.to(device)
           out = cls(upstream(feats))
           prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
           probs.append(prob); labels.append(emo.numpy())
   probs = np.concatenate(probs); labels = np.concatenate(labels)
   acc = accuracy_score(labels, probs > 0.5)
   auc = roc_auc_score(labels, probs)
   return acc, auc

def evaluate_speaker_leakage(upstream, loader):
   upstream.eval()
   preds, labels = [], []
   with torch.no_grad():
       for feats, _, spk in loader:
           feats = feats.to(device)
           out = speaker_cls(upstream(feats))
           pred = out.argmax(1).cpu().numpy()
           preds.append(pred); labels.append(spk.numpy())
   preds = np.concatenate(preds); labels = np.concatenate(labels)
   return accuracy_score(labels, preds)

In [ ]:
num_speakers = df_copd_asthma_speech['sid'].nunique()
num_speakers

In [ ]:
# 1. Load the model that performs best on the validation set.


# =========Initialize the model=========
upstream = UpstreamModel().to(device)
emotion_cls = EmotionClassifier().to(device)

best_checkpoint = torch.load("20260807_task1a_lambda0.05.pth")
upstream.load_state_dict(best_checkpoint['upstream'])
emotion_cls.load_state_dict(best_checkpoint['emotion_cls'])

# 2. A final, one-time evaluation was performed on the test set.
final_test_acc, final_test_auc = evaluate_emotion(upstream, emotion_cls, test_loader)
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"Final Test AUC: {final_test_auc:.4f}")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
)


def collect_predictions(upstream_model, emotion_classifier, data_loader):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    upstream_model.to(device)
    emotion_classifier.to(device)

    upstream_model.eval()
    emotion_classifier.eval()

    probs_list = []
    labels_list = []

    with torch.no_grad():
        for features, emotion_labels, _ in data_loader:
            features = features.to(device)

            embeddings = upstream_model(features)
            outputs = emotion_classifier(embeddings)

            # Assume category 1 is the positive class.
            probs = F.softmax(outputs, dim=1)[:, 1]

            probs_list.append(probs.detach().cpu().numpy())
            labels_list.append(emotion_labels.detach().cpu().numpy())

    if len(probs_list) == 0:
        return np.array([]), np.array([])

    all_probs = np.concatenate(probs_list, axis=0)
    all_labels = np.concatenate(labels_list, axis=0)

    return all_probs, all_labels


def _safe_confusion_matrix(labels, preds):
    """
    Securely compute the binary classification confusion matrix, guaranteeing a return shape of (2, 2).
    """
    if len(labels) == 0:
        return np.zeros((2, 2), dtype=np.int64)

    return confusion_matrix(labels, preds, labels=[0, 1])


def _threshold_independent_metrics(labels, probs):
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return {
            "auc": float("nan"),
            "pr_auc": float("nan"),
        }

    auc = roc_auc_score(labels, probs)
    pr_auc = average_precision_score(labels, probs)

    return {
        "auc": float(auc),
        "pr_auc": float(pr_auc),
    }


def evaluate_at_threshold(labels, probs, threshold):
    """
    Evaluate the metric at a given threshold.

    This can be used for `val` or `test`.
    """
    labels = np.asarray(labels).astype(np.int64)
    probs = np.asarray(probs).astype(np.float64)

    if len(labels) == 0:
        cm = np.zeros((2, 2), dtype=np.int64)
        return {
            "threshold": float(threshold),
            "acc": 0.0,
            "auc": float("nan"),
            "pr_auc": float("nan"),
            "f1": 0.0,
            "macro_f1": 0.0,
            "sensitivity": 0.0,
            "specificity": 0.0,
            "balanced_acc": 0.0,
            "cm": cm,
        }

    preds = (probs >= threshold).astype(np.int64)

    cm = _safe_confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    n = len(labels)

    acc = (tp + tn) / n if n > 0 else 0.0

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    balanced_acc = (sensitivity + specificity) / 2.0

    f1 = f1_score(labels, preds, zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    threshold_independent = _threshold_independent_metrics(labels, probs)

    return {
        "threshold": float(threshold),
        "acc": float(acc),
        "auc": threshold_independent["auc"],
        "pr_auc": threshold_independent["pr_auc"],
        "f1": float(f1),
        "macro_f1": float(macro_f1),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "balanced_acc": float(balanced_acc),
        "cm": cm,
    }


def select_threshold(
    labels,
    probs,
    min_sensitivity=0.75,
    min_specificity=0.75,
):
    """
    Select the optimal threshold on the validation set.

    Selection logic:

    1. Prioritize satisfying:

    sensitivity >= min_sensitivity

    specificity >= min_specificity

    2. Among the thresholds that satisfy these conditions, select the one with the highest F1 score.

    3. If no threshold satisfies both conditions simultaneously, revert to satisfying only sensitivity.

    4. If still no threshold satisfies both conditions, select the threshold with the highest global F1 score.
    """
    labels = np.asarray(labels).astype(np.int64)
    probs = np.asarray(probs).astype(np.float64)

    if len(labels) == 0:
        print("Warning: validation set is empty. Use threshold=0.5.")
        return 0.5

    if len(np.unique(labels)) < 2:
        print(
            "Warning: validation set only contains one class. "
            "Cannot use ROC curve to select threshold. Use threshold=0.5."
        )
        return 0.5

    # ROC curve gives all candidate thresholds
    _, _, thresholds = roc_curve(labels, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)

    # Remove inf / nan
    thresholds = thresholds[np.isfinite(thresholds)]

    # Manually add 0 and 1 to ensure that the candidate threshold covers the boundary conditions.
    thresholds = np.unique(
        np.concatenate(
            [
                thresholds,
                np.array([0.0, 1.0], dtype=np.float64),
            ]
        )
    )

    # The probability threshold is typically limited to [0, 1].
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    if thresholds.size == 0:
        thresholds = np.array([0.5], dtype=np.float64)

    f1_scores = []
    sens_scores = []
    spec_scores = []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)

        f1_scores.append(
            float(f1_score(labels, preds, zero_division=0))
        )

        cm = _safe_confusion_matrix(labels, preds)
        tn, fp, fn, tp = cm.ravel()

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        sens_scores.append(sens)
        spec_scores.append(spec)

    # Thresholds that simultaneously satisfy sensitivity and specificity
    valid_indices = [
        i
        for i in range(len(thresholds))
        if sens_scores[i] >= min_sensitivity
        and spec_scores[i] >= min_specificity
    ]

    if valid_indices:
        best_idx = valid_indices[
            int(np.argmax([f1_scores[i] for i in valid_indices]))
        ]
    else:
        print(
            "Warning: No threshold meets both constraints. "
            "Falling back to Sensitivity-only constraint."
        )

        valid_sens_indices = [
            i
            for i, s in enumerate(sens_scores)
            if s >= min_sensitivity
        ]

        if valid_sens_indices:
            best_idx = valid_sens_indices[
                int(np.argmax([f1_scores[i] for i in valid_sens_indices]))
            ]
        else:
            print(
                "Warning: No threshold meets even Sensitivity constraint. "
                "Fallback to global best F1 threshold."
            )
            best_idx = int(np.argmax(f1_scores))

    best_threshold = float(thresholds[best_idx])

    return best_threshold


def evaluate_emotion_classifier(
    upstream_model,
    emotion_classifier,
    val_loader,
    test_loader,
    min_sensitivity=0.75,
    min_specificity=0.75,
):
    """
    Evaluation process without test set leakage:

    1. Collect the predicted probabilities of the validation set using `val_loader`;

    2. Select the optimal threshold on the `val` set;

    3. Fix the threshold;

    4. Evaluate the test set performance again using `test_loader`.

    Note:
    - `val_loader` cannot be `test_loader`;

    - The `val` set should come from a training set partition or an independent validation set;

    - The `test` set can only be used once in the final evaluation.
    """
    if val_loader is None:
        raise ValueError(
            "Please provide val_loader. "
            "Threshold selection should be done on validation set, not test set."
        )

    print("=" * 70)
    print("Step 1: Reason on the value set and select a threshold ...")
    print("=" * 70)

    val_probs, val_labels = collect_predictions(
        upstream_model,
        emotion_classifier,
        val_loader,
    )

    best_threshold = select_threshold(
        val_labels,
        val_probs,
        min_sensitivity=min_sensitivity,
        min_specificity=min_specificity,
    )

    print(f"The optimal threshold selected based on the val set.: {best_threshold:.4f}")

    # The results on the validation set are for diagnostic reference only.
    val_default = evaluate_at_threshold(val_labels, val_probs, 0.5)
    val_best = evaluate_at_threshold(val_labels, val_probs, best_threshold)

    print()
    print("=" * 70)
    print("Step 2: Evaluate on the test set using a fixed threshold. ...")
    print("=" * 70)

    test_probs, test_labels = collect_predictions(
        upstream_model,
        emotion_classifier,
        test_loader,
    )

    # The default threshold for the test set is 0.5.
    test_default = evaluate_at_threshold(test_labels, test_probs, 0.5)

    # The test set uses the optimal threshold selected by val.
    test_best = evaluate_at_threshold(test_labels, test_probs, best_threshold)


    return {
        # ===============================
        # test set metrics at the default threshold of 0.5
        # ===============================
        "acc": test_default["acc"],
        "auc": test_default["auc"],
        "pr_auc": test_default["pr_auc"],
        "f1_05": test_default["f1"],
        "sensitivity": test_default["sensitivity"],
        "specificity": test_default["specificity"],
        "balanced_acc_05": test_default["balanced_acc"],
        "cm": test_default["cm"],

        # ===============================
        # test set metrics at the best threshold selected by val
        # ===============================
        "best_threshold": best_threshold,
        "best_f1": test_best["f1"],
        "best_acc": test_best["acc"],
        "best_sensitivity": test_best["sensitivity"],
        "best_specificity": test_best["specificity"],
        "best_balanced_acc": test_best["balanced_acc"],
        "best_cm": test_best["cm"],
        "best_macro_f1": test_best["macro_f1"],

        # ===============================
        # test set positive class probabilities for further analysis
        "prob": test_probs,

        # ===============================
        # validation set metrics for diagnostic reference
        # ===============================
        "val_default": val_default,
        "val_best": val_best,
        "test_default": test_default,
        "test_best": test_best,
    }

In [ ]:
results = evaluate_emotion_classifier(
    upstream_model=upstream,
    emotion_classifier=emotion_cls,
    val_loader=val_loader,
    test_loader=test_loader,
    min_sensitivity=0.8,
    min_specificity=0.7,
)

In [ ]:
print()
print("=" * 70)
print("[Final evaluation results]")
print("Note: The validation set is only used for threshold selection and should not be used for test set evaluation.")
print("=" * 70)

print()
print("The default threshold for the test set is 0.5.:")
print(f"  ACC: {results['acc']:.4f}")
print(f"  AUC: {results['auc']:.4f}")
print(f"  PR-AUC: {results['pr_auc']:.4f}")
print(f"  Sensitivity: {results['sensitivity']:.4f}")
print(f"  Specificity: {results['specificity']:.4f}")
print(f"  F1-score: {results['f1_05']:.4f}")
print(f"  Balanced Accuracy: {results['balanced_acc_05']:.4f}")
print(f"  Confusion Matrix:\n{results['cm']}")

print()
print(f"The best threshold selected by applying val to the test set. ({results['best_threshold']:.4f}):")
print(f"  ACC: {results['best_acc']:.4f}")
print(f"  Sensitivity: {results['best_sensitivity']:.4f}")
print(f"  Specificity: {results['best_specificity']:.4f}")
print(f"  F1-score: {results['best_f1']:.4f}")
print(f"  Macro F1-score: {results['best_macro_f1']:.4f}")
print(f"  Balanced Accuracy: {results['best_balanced_acc']:.4f}")
print(f"  Confusion Matrix:\n{results['best_cm']}")

print()
print("Validation set reference results (for threshold selection only):")
print(" Default threshold: 0.5:")
print(f"    ACC: {results['val_default']['acc']:.4f}")
print(f"    Sensitivity: {results['val_default']['sensitivity']:.4f}")
print(f"    Specificity: {results['val_default']['specificity']:.4f}")
print(f"    F1-score: {results['val_default']['f1']:.4f}")

print()
print(" Results after selecting a threshold on the validation set:")
print(f"    Threshold: {results['val_best']['threshold']:.4f}")
print(f"    ACC: {results['val_best']['acc']:.4f}")
print(f"    Sensitivity: {results['val_best']['sensitivity']:.4f}")
print(f"    Specificity: {results['val_best']['specificity']:.4f}")
print(f"    F1-score: {results['val_best']['f1']:.4f}")

In [ ]:
print(f"\n best threshold ({results['best_threshold']:.50f}):")

In [ ]:
prob_adv = results['prob']

In [ ]:
emb_test, spk_test = extract_embeddings(upstream, test_loader, device)
j_ratio = compute_j_ratio(emb_test, spk_test)
j_ratio

In [ ]:
emb_adv = emb_test

In [ ]:
# 1. Load the model that performs best on the validation set.


# ========= Initialize the model =========
upstream = UpstreamModel().to(device)
emotion_cls = EmotionClassifier().to(device)

best_checkpoint = torch.load("*****.pth")
upstream.load_state_dict(best_checkpoint['upstream'])
emotion_cls.load_state_dict(best_checkpoint['emotion_cls'])

# 2. A final, one-time evaluation was performed on the test set.
final_test_acc, final_test_auc = evaluate_emotion(upstream, emotion_cls, test_loader)
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"Final Test AUC: {final_test_auc:.4f}")


In [ ]:
results = evaluate_emotion_classifier(
    upstream_model=upstream,
    emotion_classifier=emotion_cls,
    val_loader=val_loader,
    test_loader=test_loader,
    min_sensitivity=0.9,
    min_specificity=0.6,
)

In [ ]:
print()
print("=" * 70)
print("[Final evaluation results]")
print("Note: The optimal threshold comes from the val set; the test set is not involved in threshold selection.")
print("=" * 70)

print()
print("Default threshold for test set(0.5):")
print(f"  ACC: {results['acc']:.4f}")
print(f"  AUC: {results['auc']:.4f}")
print(f"  PR-AUC: {results['pr_auc']:.4f}")
print(f"  Sensitivity: {results['sensitivity']:.4f}")
print(f"  Specificity: {results['specificity']:.4f}")
print(f"  F1-score: {results['f1_05']:.4f}")
print(f"  Balanced Accuracy: {results['balanced_acc_05']:.4f}")
print(f"  Confusion Matrix:\n{results['cm']}")

print()
print(f"The best threshold selected by applying val to the test set. ({results['best_threshold']:.4f}):")
print(f"  ACC: {results['best_acc']:.4f}")
print(f"  Sensitivity: {results['best_sensitivity']:.4f}")
print(f"  Specificity: {results['best_specificity']:.4f}")
print(f"  F1-score: {results['best_f1']:.4f}")
print(f"  Macro F1-score: {results['best_macro_f1']:.4f}")
print(f"  Balanced Accuracy: {results['best_balanced_acc']:.4f}")
print(f"  Confusion Matrix:\n{results['best_cm']}")

print()
print("Validation set reference results (for threshold selection only):")
print("  Default threshold: 0.5:")
print(f"    ACC: {results['val_default']['acc']:.4f}")
print(f"    Sensitivity: {results['val_default']['sensitivity']:.4f}")
print(f"    Specificity: {results['val_default']['specificity']:.4f}")
print(f"    F1-score: {results['val_default']['f1']:.4f}")

print()
print("  Results after selecting a threshold on the validation set:")
print(f"    Threshold: {results['val_best']['threshold']:.4f}")
print(f"    ACC: {results['val_best']['acc']:.4f}")
print(f"    Sensitivity: {results['val_best']['sensitivity']:.4f}")
print(f"    Specificity: {results['val_best']['specificity']:.4f}")
print(f"    F1-score: {results['val_best']['f1']:.4f}")

In [ ]:
print(f"\n best threshold ({results['best_threshold']:.50f}):")

In [ ]:
emb_test, spk_test = extract_embeddings(upstream, test_loader, device)
j_ratio = compute_j_ratio(emb_test, spk_test)
j_ratio

In [ ]:
emb_baseline = emb_test

In [ ]:
prob_baseline = results['prob']

In [ ]:
len(prob_baseline)

In [ ]:
import numpy as np

def compute_j_ratio(embeddings, labels):
    classes = np.unique(labels)
    if len(classes) < 2:
        return 0.0

    D = embeddings.shape[1]
    overall_mean = np.mean(embeddings, axis=0)

    S_B = np.zeros((D, D))
    S_W = np.zeros((D, D))

    for c in classes:
        X_c = embeddings[labels == c]
        mean_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]

        # 类间散度
        diff = (mean_c - overall_mean).reshape(-1, 1)
        S_B += n_c * diff @ diff.T

        # Intra-class divergence to prevent np.cov from generating NaN when n_c=1.
        if n_c < 2:
            cov = np.zeros((D, D))
        else:
            cov = np.cov(X_c, rowvar=False)

        S_W += (n_c - 1) * cov

    return np.trace(S_B) / (np.trace(S_W) + 1e-8)


# ============================================================
# 1. data preparation
# ============================================================
emb_baseline = np.array(emb_baseline)   # (N, D)
emb_ours     = np.array(emb_adv)        # (N, D)
spk_labels   = np.array(spk_test)       # (N,)

assert len(spk_labels) == emb_baseline.shape[0] == emb_ours.shape[0], \
    "Sample size mismatch!"

# ============================================================
# 2. Hypothesis setting
# ============================================================
# "less"      H1: ours < baseline, i.e., J-Ratio decreases
# "greater"   H1: ours > baseline, i.e., J-Ratio increases
ALTERNATIVE = "less"

rng = np.random.RandomState(42)
n_boot = 5000

N = len(spk_labels)
print(f"sample size: {N}, Embedded Dimension: {emb_baseline.shape[1]}")
print(f"Number of speaker categories: {len(np.unique(spk_labels))}")
print("=" * 60)


# ============================================================
# 3. Paired Bootstrap Resampling
# ============================================================
boot_b = []
boot_o = []

for _ in range(n_boot):
    idx = rng.choice(N, N, replace=True)

    # ensure that each sampled speaker has at least 2 samples to prevent np.cov from producing NaN
    if len(np.unique(spk_labels[idx])) < 2:
        continue

    _, counts = np.unique(spk_labels[idx], return_counts=True)
    if np.any(counts < 2):
        continue

    j_b = compute_j_ratio(emb_baseline[idx], spk_labels[idx])
    j_o = compute_j_ratio(emb_ours[idx], spk_labels[idx])

    if not (np.isfinite(j_b) and np.isfinite(j_o)):
        continue

    boot_b.append(j_b)
    boot_o.append(j_o)


boot_b = np.array(boot_b)
boot_o = np.array(boot_o)

if len(boot_b) == 0:
    raise RuntimeError("No valid Bootstrap sample is available. Please check the data or speaker tags")

boot_diff = boot_o - boot_b


# ============================================================
# 4. Observed J-Ratio and its difference
# ============================================================
obs_j_b = compute_j_ratio(emb_baseline, spk_labels)
obs_j_o = compute_j_ratio(emb_ours, spk_labels)
obs_diff = obs_j_o - obs_j_b


# ============================================================
# 5. Two-sided 95% Confidence Interval
# ============================================================
ci_b_low, ci_b_high = np.percentile(boot_b, [2.5, 97.5])
ci_o_low, ci_o_high = np.percentile(boot_o, [2.5, 97.5])

print("\nJ-Ratio point estimates and two-sided 95% CI:")
print(f"  Baseline: {obs_j_b:.2f}  [{ci_b_low:.2f}, {ci_b_high:.2f}]")
print(f"  Ours:     {obs_j_o:.2f}  [{ci_o_low:.2f}, {ci_o_high:.2f}]")


# ============================================================
# 6. One-sided Bootstrap Significance Test
# ============================================================
# Compute the 90% confidence interval for the difference
ci90_low, ci90_high = np.percentile(boot_diff, [5, 95])

if ALTERNATIVE == "less":
    # H1: ours < baseline， Δ = ours - baseline < 0
    hypothesis = "Ours < Baseline"

    # One-sided p-value: The proportion of bootstrap differences greater than or equal to 0.
    # Add 1 for smoothing, to avoid p=0
    p_one_sided = (np.sum(boot_diff >= 0) + 1) / (len(boot_diff) + 1)

    # 95% upper bound on one side
    one_sided_bound = np.percentile(boot_diff, 95)

    bound_name = "one-sided 95% upper bound"
    is_significant = one_sided_bound < 0

elif ALTERNATIVE == "greater":
    # H1: ours > baseline， Δ = ours - baseline > 0
    hypothesis = "Ours > Baseline"

    # One-sided p-value: The proportion of bootstrap differences less than or equal to 0.
    # Add 1 for smoothing, to avoid p=0
    p_one_sided = (np.sum(boot_diff <= 0) + 1) / (len(boot_diff) + 1)

    # 95% lower bound on one side
    one_sided_bound = np.percentile(boot_diff, 5)

    bound_name = "one-sided 95% lower bound"
    is_significant = one_sided_bound > 0

else:
    raise ValueError("ALTERNATIVE can only be 'less' or 'greater'。")


print("\n" + "=" * 70)
print("One-sided Paired Bootstrap Significance Test")
print("=" * 70)
print(f"  Alternative hypothesis      = {hypothesis}")
print(f"  ΔJ-Ratio (Ours - Baseline)  = {obs_diff:+.2f}")
print(f"  90% CI of ΔJ-Ratio          = [{ci90_low:+.2f}, {ci90_high:+.2f}]")
print(f"  {bound_name:<31}= {one_sided_bound:+.2f}")
print(f"  one-sided p-value           = {p_one_sided:.2f}")
print("-" * 70)

if is_significant:
    print(" Conclusion: The 95% confidence interval for one side does not include 0, and is significant on one side.")
else:
    print(" Conclusion: The 95% confidence interval for one side includes 0, and the result is not significant on one side.")

if p_one_sided < 0.05:
    print(" Conclusion: one-sided p < 0.05, statistically significant.")
elif p_one_sided < 0.1:
    print(" Conclusion: one-sided p < 0.1, indicating significant margins.")
else:
    print(" Conclusion: one-sided p >= 0.1, not significant.")

In [ ]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    f1_score,
)

# ============================================================
# 0. data preparation
# ============================================================
y_true          = np.asarray(y_emo_test).astype(int)
prob_baseline   = np.asarray(prob_baseline).astype(float)
prob_ours       = np.asarray(prob_adv).astype(float)

assert y_true.ndim == 1
assert prob_baseline.shape == prob_ours.shape == y_true.shape
assert set(np.unique(y_true)).issubset({0, 1})

threshold_baseline = 0.00000000000000052977061587133455339282406271195214
threshold_ours     = 0.04375078529119491577148437500000000000000000000000

rng = np.random.RandomState(42)
n_boot = 5000

metric_names = [
    "AUC",
    "PR-AUC",
    "Recall_Class1",
    "Recall_Class0",
    "Balanced_Accuracy",
    "Macro_F1",
]

# ============================================================
# 1. single model metrics
# ============================================================
def single_model_metrics(y, prob, threshold):
    pred = (prob >= threshold).astype(int)

    if len(np.unique(y)) < 2:
        auc = np.nan
        prauc = np.nan
    else:
        auc = roc_auc_score(y, prob)
        prauc = average_precision_score(y, prob)

    rec1 = recall_score(y, pred, pos_label=1, zero_division=0)
    rec0 = recall_score(y, pred, pos_label=0, zero_division=0)

    bal_acc = (rec0 + rec1) / 2.0

    macro_f1 = f1_score(
        y,
        pred,
        average="macro",
        zero_division=0
    )

    return np.array(
        [auc, prauc, rec1, rec0, bal_acc, macro_f1],
        dtype=float
    )


# ============================================================
# 2. Stratified bootstrap index
# ============================================================
def stratified_bootstrap_index(y, rng):
    idx_all = []
    for c in np.unique(y):
        idx_c = np.flatnonzero(y == c)
        idx_all.append(
            rng.choice(idx_c, size=len(idx_c), replace=True)
        )
    idx = np.concatenate(idx_all)
    rng.shuffle(idx)
    return idx


# ============================================================
# 3. Observed metrics
# ============================================================
obs_b = single_model_metrics(
    y_true,
    prob_baseline,
    threshold_baseline
)

obs_o = single_model_metrics(
    y_true,
    prob_ours,
    threshold_ours
)

obs_diff = obs_o - obs_b

# ============================================================
# 4. Bootstrap
# ============================================================
boot_b_list = []
boot_o_list = []

for _ in range(n_boot):
    idx = stratified_bootstrap_index(y_true, rng)

    yb = y_true[idx]
    pb = prob_baseline[idx]
    po = prob_ours[idx]

    mb = single_model_metrics(yb, pb, threshold_baseline)
    mo = single_model_metrics(yb, po, threshold_ours)

    if np.all(np.isfinite(mb)) and np.all(np.isfinite(mo)):
        boot_b_list.append(mb)
        boot_o_list.append(mo)

boot_b = np.asarray(boot_b_list)
boot_o = np.asarray(boot_o_list)
boot_diff = boot_o - boot_b

print(f"Valid bootstrap samples: {len(boot_b)} / {n_boot}")
print("=" * 85)

# ============================================================
# 5. Confidence Interval and p-value
# ============================================================
for i, name in enumerate(metric_names):

    ci_b = np.percentile(boot_b[:, i], [2.5, 97.5])
    ci_o = np.percentile(boot_o[:, i], [2.5, 97.5])
    ci_d = np.percentile(boot_diff[:, i], [2.5, 97.5])

    # bootstrap p-value
    # a simple two-sided tail probability
    if np.isclose(obs_diff[i], 0.0, atol=1e-12):
        p_value = 1.0
    elif obs_diff[i] > 0:
        tail_count = np.sum(boot_diff[:, i] <= 0)
        p_value = min(1.0, 2.0 * (tail_count + 1) / (len(boot_diff) + 1))
    else:
        tail_count = np.sum(boot_diff[:, i] >= 0)
        p_value = min(1.0, 2.0 * (tail_count + 1) / (len(boot_diff) + 1))

    if ci_d[0] > 0 or ci_d[1] < 0:
        sig = "Significant"
    else:
        sig = "Not significant"

    print(name)
    print(f"  Baseline: {obs_b[i]:.2f}   95% CI [{ci_b[0]:.2f}, {ci_b[1]:.2f}]")
    print(f"  Ours:     {obs_o[i]:.2f}   95% CI [{ci_o[0]:.2f}, {ci_o[1]:.2f}]")
    print(f"  Δ Ours-Baseline: {obs_diff[i]:+.2f}   95% CI [{ci_d[0]:+.2f}, {ci_d[1]:+.2f}]")
    print(f"  p ≈ {p_value:.2f}   decision: {sig}")
    print("-" * 85)